# Noise Monitoring with K-Means: Training Notebook

**Goal:** understand the supplied dataset, train a K-Means model, give the three clusters relative names (Quiet, Moderate, Noisy), evaluate the grouping carefully, and save the model for the Streamlit dashboard.

> This is an unsupervised-learning task. There are no independent ground-truth labels in the supplied data, so a conventional classification accuracy score would be misleading. The category names are assigned after clustering by sorting clusters by their average estimated digital level.

## 1. Set up the environment

Install the project dependencies before running the notebook. From the project folder, run `pip install -r requirements.txt`. If Jupyter is not installed, also run `pip install notebook matplotlib`.

In [ ]:
from pathlib import Path
from datetime import datetime
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
N_CLUSTERS = 3
FEATURE_COLUMNS = [
    "estimated_level_dbfs",
    "peak_amplitude",
    "zero_crossing_rate",
    "spectral_centroid_hz",
]
CATEGORY_NAMES = ["Quiet", "Moderate", "Noisy"]

# Notebook is in notebooks/, so project root is its parent directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "noise_data.csv"
MODEL_PATH = ROOT / "models" / "kmeans_noise_model.joblib"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT.resolve())
print("Dataset exists:", DATA_PATH.exists())

## 2. Load and inspect the dataset

Each row represents one audio measurement window. The supplied CSV includes time and location metadata, acoustic features, and previous cluster/category labels. For training, we use the four numeric features listed below; timestamps and location are metadata rather than model inputs.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nExisting category counts (these are stored labels, not ground truth):")
display(df["noise_category"].value_counts(dropna=False))

## 3. Understand the features

- **Estimated level (dBFS):** logarithmic RMS amplitude relative to digital full scale. It is not calibrated sound pressure level (dB SPL).
- **Peak amplitude:** largest absolute sample amplitude in the audio window.
- **Zero-crossing rate:** fraction of adjacent samples that change sign; it can relate to signal noisiness and frequency content.
- **Spectral centroid (Hz):** magnitude-weighted average frequency; often described as a measure of spectral brightness.

RMS is present in the dataset and is useful for exploration, but the original model uses the four features above. We retain that feature definition so the notebook and dashboard match the existing project.

In [ ]:
missing_feature_rows = df[FEATURE_COLUMNS].isna().any(axis=1).sum()
print(f"Rows with at least one missing model feature: {missing_feature_rows}")

# Keep complete numeric feature rows for training.
training_df = df.dropna(subset=FEATURE_COLUMNS).copy()
X = training_df[FEATURE_COLUMNS].astype(float)

print("Rows available for training:", len(X))
display(X.describe().T)

## 4. Explore the data visually

These plots describe the measurements before training. The existing category labels are shown only as a historical reference; they are not independent answers against which to score the model.

In [ ]:
fig, ax = plt.subplots()
training_df["estimated_level_dbfs"].hist(ax=ax, bins=20)
ax.set_title("Distribution of estimated digital level")
ax.set_xlabel("Estimated level (dBFS)")
ax.set_ylabel("Number of measurements")
plt.show()

In [ ]:
category_counts = training_df["noise_category"].value_counts()
fig, ax = plt.subplots()
category_counts.reindex(CATEGORY_NAMES, fill_value=0).plot(kind="bar", ax=ax)
ax.set_title("Stored category counts in supplied CSV")
ax.set_xlabel("Stored category")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
location_summary = training_df.groupby("location")["estimated_level_dbfs"].mean().sort_values()
display(location_summary.to_frame("mean_estimated_level_dbfs"))
fig, ax = plt.subplots()
location_summary.plot(kind="bar", ax=ax)
ax.set_title("Mean estimated level by location")
ax.set_xlabel("Location")
ax.set_ylabel("Mean dBFS")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Why scale the features?

K-Means groups points using distances. The input features have different units and ranges (for example, spectral centroid is measured in hertz while zero-crossing rate is usually between 0 and 1). Without scaling, a large numeric range can dominate the distance calculation.

`StandardScaler` transforms each feature to approximately zero mean and unit variance. Keeping scaling and K-Means in a single `Pipeline` ensures new measurements are transformed in the same way during prediction.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(
        n_clusters=N_CLUSTERS,
        random_state=RANDOM_STATE,
        n_init=10,
    )),
])

raw_clusters = pipeline.fit_predict(X)
training_df["raw_cluster_id"] = raw_clusters
print("Cluster sizes:")
display(training_df["raw_cluster_id"].value_counts().sort_index().to_frame("measurements"))

## 6. Give the clusters readable names

K-Means cluster IDs (0, 1, 2) are arbitrary; cluster 0 does not inherently mean Quiet. To make the dashboard easier to interpret, calculate each cluster's average estimated level and sort clusters from lowest to highest. Assign the names Quiet, Moderate, and Noisy in that order.

This is a project-specific naming rule, not a learned supervised label and not a calibrated noise-safety threshold.

In [ ]:
cluster_levels = {}
for cluster_id in range(N_CLUSTERS):
    mask = raw_clusters == cluster_id
    if not mask.any():
        raise ValueError(
            f"Cluster {cluster_id} is empty. Check the data and number of clusters."
        )
    cluster_levels[cluster_id] = float(
        training_df.loc[mask, "estimated_level_dbfs"].mean()
    )

ordered_clusters = sorted(cluster_levels, key=cluster_levels.get)
cluster_to_category = {
    int(cluster_id): CATEGORY_NAMES[rank]
    for rank, cluster_id in enumerate(ordered_clusters)
}

training_df["cluster_id"] = raw_clusters
training_df["noise_category"] = [
    cluster_to_category[int(c)] for c in raw_clusters
]

summary = pd.DataFrame([
    {
        "cluster_id": cluster_id,
        "category": cluster_to_category[cluster_id],
        "mean_estimated_level_dbfs": cluster_levels[cluster_id],
        "count": int((raw_clusters == cluster_id).sum()),
    }
    for cluster_id in ordered_clusters
])
display(summary)

## 7. Evaluate the clustering (with the right caveat)

The **silhouette score** summarizes how close observations are to their own cluster compared with other clusters. It ranges from -1 to 1; larger values often indicate more separated clusters. It does not prove that the categories are objectively correct, and it does not replace listening tests or calibrated sound measurements.

We calculate it in scaled feature space because K-Means is fit on scaled features. Silhouette score is defined only when there are between 2 and `n_samples - 1` distinct cluster labels.

In [ ]:
X_scaled = pipeline.named_steps["scaler"].transform(X)
unique_labels = np.unique(raw_clusters)

if 2 <= len(unique_labels) < len(X):
    score = silhouette_score(X_scaled, raw_clusters)
    print(f"Silhouette score: {score:.3f}")
else:
    print("Silhouette score is not defined for this cluster assignment.")

## 8. Train and save the reusable model bundle

The dashboard expects a dictionary with:
- `pipeline`: fitted scaler + K-Means model;
- `cluster_to_category`: mapping from cluster ID to readable category;
- `feature_columns`: feature order used for prediction;
- `trained_at`: timestamp of the training run.

The model file is saved in `models/`, matching the Streamlit app. The notebook also writes a CSV with the refreshed cluster assignments and category names. The original CSV is backed up first, so the supplied data is not silently overwritten.

In [ ]:
model_bundle = {
    "pipeline": pipeline,
    "cluster_to_category": cluster_to_category,
    "feature_columns": FEATURE_COLUMNS,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
}
joblib.dump(model_bundle, MODEL_PATH)
print("Saved model:", MODEL_PATH.resolve())

# Save updated training labels to a separate file first.
labelled_path = ROOT / "data" / "noise_data_labelled.csv"
labelled_df = df.copy()
labelled_df.loc[training_df.index, "cluster_id"] = training_df["cluster_id"]
labelled_df.loc[training_df.index, "noise_category"] = training_df["noise_category"]
labelled_df.to_csv(labelled_path, index=False)
print("Saved labelled data:", labelled_path.resolve())

## 9. Test prediction on one measurement

This final test loads the exact bundle structure used by the dashboard and predicts the cluster for one observation. The prediction should use the same feature names and order as training.

In [ ]:
loaded_bundle = joblib.load(MODEL_PATH)
sample = X.iloc[[0]]
predicted_cluster = int(loaded_bundle["pipeline"].predict(sample)[0])
predicted_category = loaded_bundle["cluster_to_category"][predicted_cluster]

print("Input features:")
display(sample)
print("Predicted cluster:", predicted_cluster)
print("Predicted category:", predicted_category)

## 10. What to explain in a project presentation

1. **Problem:** summarize relative noise patterns across recorded locations and times.
2. **Data:** each row is a short audio measurement represented by numeric acoustic features.
3. **Preprocessing:** remove rows missing required features and standardize features.
4. **Algorithm:** K-Means partitions the measurements into three groups by feature-space similarity.
5. **Interpretation:** order groups by average estimated dBFS and name them Quiet, Moderate, and Noisy.
6. **Deployment:** the Streamlit app reads the saved CSV and Joblib model; the notebook documents how to retrain them.
7. **Limitations:** no ground-truth categories, relative rather than calibrated noise labels, a small dataset, and no live microphone capture in the hosted dashboard.

For a stronger future evaluation, collect more varied recordings and independently labelled or calibrated reference measurements. Do not describe the silhouette score as classification accuracy.